# WAXAL ASR - Whisper Large-V3 Zero-Shot Submission

Runs **Whisper Large-V3** (1.55B params) zero-shot with beam search on test audio
(Luganda, Lingala, Shona) and generates a submission CSV for the Zindi competition.

No fine-tuning needed. Uses local parquet files from `data/` folder.

**Steps:** Install deps → Setup → Load test data → Load model → Generate submission

## 1. Install Dependencies

In [7]:
!pip install -q soundfile "datasets==3.2.0" transformers
print("Dependencies installed.")

Dependencies installed.


## 2. Setup & Imports

In [8]:
import os, csv
from pathlib import Path

import datasets
import numpy as np
import torch
import transformers
from tqdm.auto import tqdm

PROJECT_ROOT = Path(".").resolve().parent

# Load HF token from .env (gitignored)
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().strip().splitlines():
        if "=" in line and not line.startswith("#"):
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
    print("HF_TOKEN loaded from .env")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"datasets version: {datasets.__version__}")
print(f"transformers version: {transformers.__version__}")

HF_TOKEN loaded from .env
Device: cuda
datasets version: 3.2.0
transformers version: 5.13.1


## 3. Load Test Data from Local Parquet Files

Loads test parquet files from the `data/` folder. Supports multiple shards per language.

In [9]:
LANGUAGES = ["lug", "lin", "sna"]
SAMPLE_RATE = 16_000
DATA_DIR = PROJECT_ROOT / "data"

# Whisper language codes (Luganda not supported, use Swahili as closest Bantu language)
WHISPER_LANG_MAP = {
    "lug": "swahili",
    "lin": "lingala",
    "sna": "shona",
}

print("Loading test data from local parquet files...\n")

test_data = {}
for lang in LANGUAGES:
    # Find all shards: lang-test-00000.parquet, lang-test-00001.parquet, etc.
    shards = sorted(DATA_DIR.glob(f"{lang}-test-*.parquet"))
    if not shards:
        print(f"  WARNING: No parquet files found for {lang}")
        continue

    print(f"  {lang}: {len(shards)} shard(s) ...", end=" ", flush=True)
    shard_datasets = [datasets.Dataset.from_parquet(str(s)) for s in shards]

    if len(shard_datasets) == 1:
        ds = shard_datasets[0]
    else:
        ds = datasets.concatenate_datasets(shard_datasets)

    ds = ds.cast_column("audio", datasets.Audio(sampling_rate=SAMPLE_RATE))
    test_data[lang] = ds
    print(f"OK ({len(ds)} examples)")

print("\nAll test data loaded.")

Loading test data from local parquet files...

  lug: 1 shard(s) ... OK (638 examples)
  lin: 2 shard(s) ... 

Generating train split: 34 examples [00:00, 319.76 examples/s]


OK (1866 examples)
  sna: 2 shard(s) ... 

Generating train split: 153 examples [00:00, 592.86 examples/s]

OK (1749 examples)

All test data loaded.


## 4. Load Whisper Small

In [ ]:
MODEL_ID = "openai/whisper-large-v3"  # 1.55B params, much better multilingual

processor = transformers.WhisperProcessor.from_pretrained(MODEL_ID)
model = transformers.WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16
)

# Disable forced decoder IDs so we control language per-sample
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []

model = model.to(device)
model.eval()

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {MODEL_ID} ({params:.1f}M params)")
print(f"Loaded on {device}")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 5. Generate Submission CSV

Transcribes all ~4,253 test samples using Whisper Large-V3 with beam search.
Uses `language` hints and `task="transcribe"` to output in native languages.

In [ ]:
test_csv_path = PROJECT_ROOT / "Test.csv"
sample_csv_path = PROJECT_ROOT / "SampleSubmission.csv"
submission_dir = PROJECT_ROOT / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / "submission_large_v3.csv"

# Read test IDs and group by language
test_ids = []
with open(test_csv_path, "r", encoding="utf-8") as fh:
    for row in csv.DictReader(fh):
        test_ids.append(row["ID"])

lang_to_ids = {}
for tid in test_ids:
    lang = tid.split("_")[0]
    lang_to_ids.setdefault(lang, []).append(tid)

print(f"Test set: {len(test_ids)} samples")
for lang, ids in lang_to_ids.items():
    print(f"  {lang}: {len(ids)} samples")

# Run inference with language hints + beam search
predictions = {}

for lang in LANGUAGES:
    if lang not in lang_to_ids:
        print(f"\nSkipping {lang} (no test IDs)")
        continue

    ds = test_data[lang]
    needed_ids = set(lang_to_ids[lang])
    whisper_lang = WHISPER_LANG_MAP[lang]

    # Build ID lookup: dataset index -> full test ID
    id_lookup = {}
    for idx in range(len(ds)):
        raw_id = str(ds[idx]["id"])
        full_id = f"{lang}_{raw_id}" if not raw_id.startswith(lang) else raw_id
        if full_id in needed_ids:
            id_lookup[idx] = full_id

    print(f"\n{lang}: transcribing {len(id_lookup)} samples (whisper_lang={whisper_lang})...")

    for idx in tqdm(sorted(id_lookup.keys()), desc=f"Predict {lang}"):
        example = ds[idx]
        audio_array = np.asarray(example["audio"]["array"], dtype=np.float32)

        input_features = processor.feature_extractor(
            audio_array,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
        ).input_features.to(device=device, dtype=torch.float16)

        with torch.no_grad():
            pred_ids = model.generate(
                input_features,
                max_new_tokens=225,
                language=whisper_lang,
                task="transcribe",
                num_beams=5,
                no_repeat_ngram_size=3,
                condition_on_prev_tokens=False,
            )

        transcript = processor.tokenizer.decode(
            pred_ids[0], skip_special_tokens=True
        ).strip()
        predictions[id_lookup[idx]] = transcript

# Write submission CSV
with open(submission_path, "w", encoding="utf-8", newline="") as fh:
    writer = csv.writer(fh)
    writer.writerow(["ID", "Target"])
    for tid in test_ids:
        writer.writerow([tid, predictions.get(tid, "")])

print(f"\nSubmission written to: {submission_path}")
print(f"Predictions: {len(predictions)} / {len(test_ids)}")

# Validate
if sample_csv_path.exists():
    with open(sample_csv_path, "r", encoding="utf-8") as fh:
        expected_ids = {row["ID"] for row in csv.DictReader(fh)}
    with open(submission_path, "r", encoding="utf-8") as fh:
        submitted_ids = {row["ID"] for row in csv.DictReader(fh)}
    missing = expected_ids - submitted_ids
    empty = sum(1 for tid in test_ids if not predictions.get(tid, ""))
    if missing:
        print(f"WARNING: Missing {len(missing)} IDs!")
    elif empty:
        print(f"WARNING: {empty} IDs have empty transcriptions")
    else:
        print("Submission validation PASSED - all IDs present with transcriptions")